Language Modeling is the task of predicting the next word (or character) in a sequence based on the context of previous words (or characters).

In [1]:
!pip install nltk

In [2]:
import torch
import torch.nn as nn
import numpy as np
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import word_tokenize
import nltk

In [3]:
document = """About the Program
What is the course fee for  Data Science Mentorship Program (DSMP 2023)
The course follows a monthly subscription model where you have to make monthly payments of Rs 799/month.
What is the total duration of the course?
The total duration of the course is 7 months. So the total course fee becomes 799*7 = Rs 5600(approx.)
What is the syllabus of the mentorship program?
We will be covering the following modules:
Python Fundamentals
Python libraries for Data Science
Data Analysis
SQL for Data Science
Maths for Machine Learning
ML Algorithms
Practical ML
MLOPs
Case studies
You can check the detailed syllabus here - https://learnwith.campusx.in/courses/CampusX-Data-Science-Mentorship-Program-637339afe4b0615a1bbed390
Will Deep Learning and NLP be a part of this program?
No, NLP and Deep Learning both are not a part of this program’s curriculum.
What if I miss a live session? Will I get a recording of the session?
Yes all our sessions are recorded, so even if you miss a session you can go back and watch the recording.
Where can I find the class schedule?
Checkout this google sheet to see month by month time table of the course - https://docs.google.com/spreadsheets/d/16OoTax_A6ORAeCg4emgexhqqPv3noQPYKU7RJ6ArOzk/edit?usp=sharing.
What is the time duration of all the live sessions?
Roughly, all the sessions last 2 hours.
What is the language spoken by the instructor during the sessions?
Hinglish
How will I be informed about the upcoming class?
You will get a mail from our side before every paid session once you become a paid user.
Can I do this course if I am from a non-tech background?
Yes, absolutely.
I am late, can I join the program in the middle?
Absolutely, you can join the program anytime.
If I join/pay in the middle, will I be able to see all the past lectures?
Yes, once you make the payment you will be able to see all the past content in your dashboard.
Where do I have to submit the task?
You don’t have to submit the task. We will provide you with the solutions, you have to self evaluate the task yourself.
Will we do case studies in the program?
Yes.
Where can we contact you?
You can mail us at nitish.campusx@gmail.com
Payment/Registration related questions
Where do we have to make our payments? Your YouTube channel or website?
You have to make all your monthly payments on our website. Here is the link for our website - https://learnwith.campusx.in/
Can we pay the entire amount of Rs 5600 all at once?
Unfortunately no, the program follows a monthly subscription model.
What is the validity of monthly subscription? Suppose if I pay on 15th Jan, then do I have to pay again on 1st Feb or 15th Feb
15th Feb. The validity period is 30 days from the day you make the payment. So essentially you can join anytime you don’t have to wait for a month to end.
What if I don’t like the course after making the payment. What is the refund policy?
You get a 7 days refund period from the day you have made the payment.
I am living outside India and I am not able to make the payment on the website, what should I do?
You have to contact us by sending a mail at nitish.campusx@gmail.com
Post registration queries
Till when can I view the paid videos on the website?
This one is tricky, so read carefully. You can watch the videos till your subscription is valid. Suppose you have purchased subscription on 21st Jan, you will be able to watch all the past paid sessions in the period of 21st Jan to 20th Feb. But after 21st Feb you will have to purchase the subscription again.
But once the course is over and you have paid us Rs 5600(or 7 installments of Rs 799) you will be able to watch the paid sessions till Aug 2024.
Why lifetime validity is not provided?
Because of the low course fee.
Where can I reach out in case of a doubt after the session?
You will have to fill a google form provided in your dashboard and our team will contact you for a 1 on 1 doubt clearance session
If I join the program late, can I still ask past week doubts?
Yes, just select past week doubt in the doubt clearance google form.
I am living outside India and I am not able to make the payment on the website, what should I do?
You have to contact us by sending a mail at nitish.campusx@gmai.com
Certificate and Placement Assistance related queries
What is the criteria to get the certificate?
There are 2 criterias:
You have to pay the entire fee of Rs 5600
You have to attempt all the course assessments.
I am joining late. How can I pay payment of the earlier months?
You will get a link to pay fee of earlier months in your dashboard once you pay for the current month.
I have read that Placement assistance is a part of this program. What comes under Placement assistance?
This is to clarify that Placement assistance does not mean Placement guarantee. So we dont guarantee you any jobs or for that matter even interview calls. So if you are planning to join this course just for placements, I am afraid you will be disappointed. Here is what comes under placement assistance
Portfolio Building sessions
Soft skill sessions
Sessions with industry mentors
Discussion on Job hunting strategies
"""


In [4]:
# Tokenization
nltk.download("punkt")
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [5]:
# tokenize
tokens=word_tokenize(document.lower())

In [6]:
vocab={"<unk>":0}

for token in Counter(tokens).keys():
  if token not in vocab:
    vocab[token]=len(vocab)
len(vocab)

289

In [7]:
len(vocab)

289

In [8]:
# extract sentences from data

input_sentences=document.split("\n")

In [9]:
def text_to_indices(sentence, vocab):

  numerical_sentence = []

  for token in sentence:
    if token in vocab:
      numerical_sentence.append(vocab[token])
    else:
      numerical_sentence.append(vocab['<unk>'])

  return numerical_sentence


In [10]:
input_numerical_sentences = []

for sentence in input_sentences:
  input_numerical_sentences.append(text_to_indices(word_tokenize(sentence.lower()), vocab))

len(input_numerical_sentences)

78

In [11]:
training_sequence=[]
for sentence in input_numerical_sentences:
  for i in range(1,len(sentence)):
    training_sequence.append(sentence[:i+1])

In [12]:
len(training_sequence)

942

In [13]:
# Find longest sequence and
# padding should be done to make size equal.

len_list=[]
for sequence in training_sequence:
  len_list.append(len(sequence))
max(len_list)

62

In [14]:
padded_training_sequence=[]
for sequence in training_sequence:
  padded_training_sequence.append([0]*(max(len_list)-len(sequence))+sequence)

len(padded_training_sequence[12])


62

In [15]:
padded_training_sequence=torch.tensor(padded_training_sequence,dtype=torch.long)
padded_training_sequence.shape

torch.Size([942, 62])

In [16]:
X=padded_training_sequence[:,:-1]
y=padded_training_sequence[:,-1]

In [17]:
class CustomDataset(Dataset):

  def __init__(self,X,y):
    self.X=X
    self.y=y

  def __len__(self):
    return self.X.shape[0]

  def __getitem__(self, idx):
    return self.X[idx],self.y[idx]

In [18]:
dataset=CustomDataset(X,y)

In [19]:
dataloader=DataLoader(dataset,batch_size=32,shuffle=True)

In [20]:
from typing_extensions import final
class LSTMModel(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.embedding=nn.Embedding(vocab_size,100)
    self.lstm=nn.LSTM(100,150,batch_first=True) # LSTM gives three output so no sequential containers
    self.fc=nn.Linear(150,vocab_size)


  def forward(self,x):
    embedded=self.embedding(x)
    intermediate_hidden_states,(final_hidden_state,final_cell_state)=self.lstm(embedded)
    return self.fc(final_hidden_state.squeeze(0))

In [21]:
model=LSTMModel(len(vocab))

In [22]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

LSTMModel(
  (embedding): Embedding(289, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (fc): Linear(in_features=150, out_features=289, bias=True)
)

In [23]:
epochs=50
learning_rate=0.001

criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=learning_rate)

In [24]:
for epochs in range(epochs):
  total_loss=0

  for batch_x,batch_y in dataloader:
    batch_x=batch_x.to(device)
    batch_y=batch_y.to(device)

    optimizer.zero_grad()
    output=model(batch_x)
    loss=criterion(output,batch_y)
    loss.backward()
    optimizer.step()
    total_loss=total_loss+loss.item()
  print(total_loss)

166.65911293029785
146.83353328704834
135.18783521652222
121.99849057197571
109.8508825302124
98.50018358230591
87.16617250442505
76.87353706359863
67.99352037906647
59.474602460861206
52.75962853431702
45.675377368927
40.08652365207672
34.97780245542526
30.870636999607086
26.934330642223358
23.878784954547882
21.033654421567917
18.85976016521454
16.706365644931793
15.096957325935364
13.605626732110977
12.540525615215302
11.467053055763245
10.495843023061752
9.73840081691742
9.094354793429375
8.667904362082481
7.9819143414497375
7.653783336281776
7.350134789943695
7.011913016438484
6.762388527393341
6.287140227854252
6.1305831372737885
5.983968712389469
5.712406352162361
5.483758084475994
5.439976774156094
5.290879346430302
5.1462347991764545
5.0741047486662865
4.78464699909091
4.779524561017752
4.759753223508596
4.687262661755085
4.489488951861858
4.451421830803156
4.355317022651434
4.347291097044945


In [29]:
# prediction

def prediction(model,vocab,text):
  #numerical indices
  tokenized_text=word_tokenize(text.lower())
  numerical_text=text_to_indices(tokenized_text,vocab)

  # padding
  padded_text=torch.tensor([0]*(61-len(numerical_text))+numerical_text,dtype=torch.long).unsqueeze(0)
  padded_text = padded_text.to(device)

  # send to model
  output=model(padded_text)

  # predicted index
  value,index=torch.max(output,dim=1)

  # merge with text
  return text+ " " +list(vocab.keys())[index]


In [30]:
prediction(model,vocab,"the course follows a")

'the course follows a monthly'

In [31]:
num_tokens=10
input_text=" The course follows a monthly"
for i in range(num_tokens):
  output_text=prediction(model,vocab,input_text)
  print(output_text)
  input_text=output_text

 The course follows a monthly subscription
 The course follows a monthly subscription model
 The course follows a monthly subscription model where
 The course follows a monthly subscription model where you
 The course follows a monthly subscription model where you have
 The course follows a monthly subscription model where you have to
 The course follows a monthly subscription model where you have to make
 The course follows a monthly subscription model where you have to make monthly
 The course follows a monthly subscription model where you have to make monthly payments
 The course follows a monthly subscription model where you have to make monthly payments of
